# Notebook 01 (Real) - Thu thap du lieu nha dat Ha Noi

Muc tieu:
- Thu thap du lieu that ve nha dat khu vuc Ha Noi.
- Tang quy mo thu thap (de sau cleaning con du mau train).
- Luu du lieu tho vao `data/raw/housing_raw_vn_hanoi_deep.csv`.

## 1) Cai dat thu vien (neu chua co)

Neu bi bao thieu thu vien, bo comment dong lenh ben duoi va chay lai.

In [16]:
%pip install requests beautifulsoup4 lxml pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [17]:
from pathlib import Path
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, RAW_DIR

(WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha'),
 WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/data/raw'))

## 2) Cau hinh crawl

In [18]:
TARGET_ROWS = 4000
PAGE_SIZE = 50
MAX_PAGES = 120

# API public listing cua Cho Tot (nha dat)
API_URL = "https://gateway.chotot.com/v1/public/ad-listing"

# Ma khu vuc Ha Noi thuong dung tren endpoint listing
HANOI_REGION_V2 = 12000

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.chotot.com/",
    "Origin": "https://www.chotot.com"
}

TARGET_ROWS, PAGE_SIZE, MAX_PAGES

(4000, 50, 120)

## 3) Ham ho tro boc tach du lieu

In [19]:
def pick_text(node, selectors):
    for css in selectors:
        found = node.select_one(css)
        if found:
            txt = found.get_text(" ", strip=True)
            if txt:
                return txt
    return None


def pick_href(node, selectors):
    for css in selectors:
        found = node.select_one(css)
        if found and found.has_attr("href"):
            href = found["href"].strip()
            if href:
                if href.startswith("/"):
                    return f"https://batdongsan.com.vn{href}"
                return href
    return None

## 4) Crawl danh sach tin

In [20]:
rows = []
seen_ids = set()
session = requests.Session()

for page in range(1, MAX_PAGES + 1):
    if len(rows) >= TARGET_ROWS:
        break

    offset = (page - 1) * PAGE_SIZE
    params = {
        "region_v2": HANOI_REGION_V2,
        "cg": 1020,  # nha dat
        "limit": PAGE_SIZE,
        "o": offset,
        "st": "s,k",
        "sp": 0,
    }

    try:
        resp = session.get(API_URL, params=params, headers=HEADERS, timeout=25)
        if resp.status_code != 200:
            print(f"[Page {page}] status {resp.status_code} -> bo qua")
            continue

        payload = resp.json()
        ads = payload.get("ads", [])
    except Exception as e:
        print(f"[Page {page}] loi request/json: {e}")
        continue

    print(f"[Page {page}] tim thay {len(ads)} tin")

    if not ads:
        continue

    for ad in ads:
        ad_id = ad.get("list_id") or ad.get("ad_id")
        if ad_id in seen_ids:
            continue
        seen_ids.add(ad_id)

        district = ad.get("district_name")
        ward = ad.get("ward_name")
        location = ", ".join([x for x in [ward, district, "Ha Noi"] if x])

        rows.append({
            "source": "chotot",
            "city": "Ha Noi",
            "price_raw": ad.get("price_string") or ad.get("price"),
            "area_raw": ad.get("size") or ad.get("area"),
            "location_raw": location if location else None,
            "detail_url": ad.get("webp_image") or ad.get("image") or None,
            "list_id": ad_id,
            "category_name": ad.get("category_name"),
            "crawl_page": page,
            "crawl_ts": pd.Timestamp.now(tz="Asia/Ho_Chi_Minh")
        })

        if len(rows) >= TARGET_ROWS:
            break

    time.sleep(random.uniform(0.8, 1.6))

print(f"\nTong so ban ghi thu duoc: {len(rows)}")

[Page 1] tim thay 50 tin
[Page 2] tim thay 50 tin
[Page 3] tim thay 50 tin
[Page 4] tim thay 50 tin
[Page 5] tim thay 50 tin
[Page 6] tim thay 50 tin
[Page 7] tim thay 50 tin
[Page 8] tim thay 50 tin
[Page 9] tim thay 50 tin
[Page 10] tim thay 50 tin
[Page 11] tim thay 50 tin
[Page 12] tim thay 50 tin
[Page 13] tim thay 50 tin
[Page 14] tim thay 50 tin
[Page 15] tim thay 50 tin
[Page 16] tim thay 50 tin
[Page 17] tim thay 50 tin
[Page 18] tim thay 50 tin
[Page 19] tim thay 50 tin
[Page 20] tim thay 50 tin
[Page 21] tim thay 50 tin
[Page 22] tim thay 50 tin
[Page 23] tim thay 50 tin
[Page 24] tim thay 50 tin
[Page 25] tim thay 50 tin
[Page 26] tim thay 50 tin
[Page 27] tim thay 50 tin
[Page 28] tim thay 50 tin
[Page 29] tim thay 50 tin
[Page 30] tim thay 50 tin
[Page 31] tim thay 50 tin
[Page 32] tim thay 50 tin
[Page 33] tim thay 50 tin
[Page 34] tim thay 50 tin
[Page 35] tim thay 50 tin
[Page 36] tim thay 50 tin
[Page 37] tim thay 50 tin
[Page 38] tim thay 50 tin
[Page 39] tim thay 50

## 5) Enrich truong sau tu endpoint chi tiet (phap ly, phong, noi that...)

In [21]:
# Tao dataframe listing ban dau
ndf = pd.DataFrame(rows)
print("Listing shape:", ndf.shape)

# Lay chi tiet tung tin theo list_id de bo sung truong sau
DETAIL_API_TEMPLATE = "https://gateway.chotot.com/v1/public/ad-listing/{list_id}"

# Gioi han so tin enrich de test nhanh. Dat = len(ndf) de enrich full 600
ENRICH_LIMIT = len(ndf)

detail_rows = []
for i, list_id in enumerate(ndf["list_id"].dropna().astype(int).head(ENRICH_LIMIT), start=1):
    try:
        detail_resp = session.get(
            DETAIL_API_TEMPLATE.format(list_id=list_id),
            headers=HEADERS,
            timeout=25,
        )
        if detail_resp.status_code != 200:
            continue

        detail_payload = detail_resp.json()
        ad = detail_payload.get("ad", {})
        params = detail_payload.get("parameters", [])
        param_map = {p.get("id"): p.get("value") for p in params if isinstance(p, dict)}

        detail_rows.append({
            "list_id": ad.get("list_id") or list_id,
            "Gia": ad.get("price"),
            "GiaText": ad.get("price_string"),
            "dienTich": ad.get("size") or ad.get("area"),
            "chieuNgang": ad.get("width"),
            "chieuDai": ad.get("length"),
            "Phongngu": ad.get("rooms"),
            "PhongTam": ad.get("toilets"),
            "SoTang": ad.get("floors"),
            "Loai": param_map.get("house_type"),
            "GiayTo": param_map.get("property_legal_document"),
            "TinhTrangNoiThat": param_map.get("furnishing_sell"),
            "Phuong": param_map.get("ward"),
            "Quan": param_map.get("area"),
            "DiaChi": param_map.get("address"),
        })

    except Exception:
        continue

    if i % 50 == 0:
        print(f"Da enrich: {i} tin")

    time.sleep(random.uniform(0.25, 0.7))

df_detail = pd.DataFrame(detail_rows).drop_duplicates(subset=["list_id"])
print("Detail shape:", df_detail.shape)

# Gop listing + detail
df_raw = ndf.merge(df_detail, on="list_id", how="left")
print("Merged shape:", df_raw.shape)
df_raw.head()

Listing shape: (2879, 10)
Da enrich: 50 tin
Da enrich: 100 tin
Da enrich: 150 tin
Da enrich: 200 tin
Da enrich: 250 tin
Da enrich: 300 tin
Da enrich: 350 tin
Da enrich: 400 tin
Da enrich: 450 tin
Da enrich: 500 tin
Da enrich: 550 tin
Da enrich: 600 tin
Da enrich: 650 tin
Da enrich: 700 tin
Da enrich: 750 tin
Da enrich: 800 tin
Da enrich: 850 tin
Da enrich: 900 tin
Da enrich: 950 tin
Da enrich: 1000 tin
Da enrich: 1050 tin
Da enrich: 1100 tin
Da enrich: 1150 tin
Da enrich: 1200 tin
Da enrich: 1250 tin
Da enrich: 1300 tin
Da enrich: 1350 tin
Da enrich: 1400 tin
Da enrich: 1450 tin
Da enrich: 1500 tin
Da enrich: 1550 tin
Da enrich: 1600 tin
Da enrich: 1650 tin
Da enrich: 1700 tin
Da enrich: 1750 tin
Da enrich: 1800 tin
Da enrich: 1850 tin
Da enrich: 1900 tin
Da enrich: 1950 tin
Da enrich: 2000 tin
Da enrich: 2050 tin
Da enrich: 2100 tin
Da enrich: 2150 tin
Da enrich: 2200 tin
Da enrich: 2250 tin
Da enrich: 2300 tin
Da enrich: 2350 tin
Da enrich: 2400 tin
Da enrich: 2450 tin
Da enrich: 250

,source,city,price_raw,area_raw,location_raw,detail_url,list_id,category_name,crawl_page,crawl_ts,...,chieuDai,Phongngu,PhongTam,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,DiaChi
0,chotot,Ha Noi,"19,8 tỷ",50.0,"Phường Láng Thượng, Ha Noi",https://cdn.chotot.com/Z4h1aUo9xoKjjsvboE08t0y...,131673394,Nhà ở,1,2026-05-01 18:14:13.228010+07:00,...,NaN,6.0,NaN,NaN,"Nhà ngõ, hẻm",NaN,NaN,Phường Láng Thượng,Quận Đống Đa,"Đường Nguyễn Chí Thanh, Phường Láng Thượng, Qu..."
1,chotot,Ha Noi,"6,35 tỷ",45.0,"Xã Hữu Hoà, Ha Noi",https://cdn.chotot.com/lTbplm1RmWfuOE6xkHLdeft...,130840917,Nhà ở,1,2026-05-01 18:14:13.230619+07:00,...,NaN,4.0,NaN,NaN,"Nhà mặt phố, mặt tiền",NaN,NaN,Xã Hữu Hoà,Huyện Thanh Trì,"Hữu lê, Xã Hữu Hoà, Huyện Thanh Trì, Hà Nội"
2,chotot,Ha Noi,6 tỷ,36.0,"Phường Hà Cầu, Ha Noi",https://cdn.chotot.com/2mDbOH9nNOXUNCDip0jto_N...,130728190,Nhà ở,1,2026-05-01 18:14:13.230619+07:00,...,NaN,4.0,NaN,NaN,"Nhà ngõ, hẻm",NaN,NaN,Phường Hà Cầu,Quận Hà Đông,"Bà triệu, Phường Hà Cầu, Quận Hà Đông, Hà Nội"
3,chotot,Ha Noi,"6,2 tỷ",34.0,"Phường Kiến Hưng, Ha Noi",https://cdn.chotot.com/STnLUpXg9zds-3dOz0lR6-1...,129837573,Nhà ở,1,2026-05-01 18:14:13.230619+07:00,...,NaN,3.0,NaN,NaN,"Nhà mặt phố, mặt tiền",Đã có sổ,NaN,Phường Kiến Hưng,Quận Hà Đông,"Đường Đa Sỹ, Phường Kiến Hưng, Quận Hà Đông, H..."
4,chotot,Ha Noi,"6,2 tỷ",36.0,"Phường Hà Cầu, Ha Noi",https://cdn.chotot.com/3Bn0lJNTz47kvy2yYEFIT2P...,131563048,Nhà ở,1,2026-05-01 18:14:13.230619+07:00,...,NaN,3.0,NaN,NaN,"Nhà mặt phố, mặt tiền",NaN,NaN,Phường Hà Cầu,Quận Hà Đông,"Phố Lê Lợi, Phường Hà Cầu, Quận Hà Đông, Hà Nội"


In [22]:
print("Shape:", df_raw.shape)
print("\nMissing values:\n", df_raw.isna().sum())

Shape: (2879, 24)

Missing values:
 source                 0
city                   0
price_raw              0
area_raw               0
location_raw           0
detail_url             4
list_id                0
category_name          0
crawl_page             0
crawl_ts               0
Gia                    1
GiaText                1
dienTich               1
chieuNgang          1529
chieuDai            1960
Phongngu               1
PhongTam            1192
SoTang               782
Loai                   1
GiayTo               660
TinhTrangNoiThat    1181
Phuong                 5
Quan                   1
DiaChi                 1
dtype: int64


## 6) Luu du lieu tho

In [23]:
raw_path = RAW_DIR / "housing_raw_vn_hanoi_deep.csv"
df_raw.to_csv(raw_path, index=False, encoding="utf-8-sig")
print(f"Da luu du lieu tho tai: {raw_path}")

Da luu du lieu tho tai: C:\Users\asus\Desktop\DuDoanGiaNha\data\raw\housing_raw_vn_hanoi_deep.csv
